================================================================
FILE : preprocess.py
WHAT THIS FILE DOES :
  Step 1 - Load the raw CSV dataset
  Step 2 - Remove useless columns
  Step 3 - Create label  0 = Benign  1 = Malicious
  Step 4 - Convert all text and hex values to numbers
  Step 5 - Fill empty cells with median
  Step 6 - Split into 80% Train and 20% Test
  Step 7 - Show train and test data sample below
  Step 8 - Save train.csv and test.csv
  Step 9 - Save class distribution plot

HOW TO RUN :  python preprocess.py
OUTPUT     :  Everything prints below. train.csv and test.csv saved.
================================================================

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
# ----------------------------------------------------------------
# FOLDER SETUP
# ----------------------------------------------------------------
os.makedirs("results/plots", exist_ok=True)
os.makedirs("data",          exist_ok=True)

In [ ]:
print("=" * 65)
print("        PREPROCESSING  —  Step 1 of 4")
print("=" * 65)

================================================================
STEP 1 : LOAD THE DATASET
================================================================
pd.read_csv reads the CSV file into a table called DataFrame
Each ROW    = one .exe file (21,752 files total)
Each COLUMN = one property of that file (77 columns)

In [ ]:
print("\n[STEP 1]  Loading dataset...")
print("         File : data/Final_Dataset_without_duplicate.csv")

In [ ]:
df = pd.read_csv("data/Final_Dataset_without_duplicate.csv")

In [ ]:
print(f"\n         Total rows (files)    : {df.shape[0]}")
print(f"         Total columns (features): {df.shape[1]}")
print(f"\n         First 3 rows of RAW DATA (before cleaning):")
print("-" * 65)
print(df.head(3).to_string())
print("-" * 65)

================================================================
STEP 2 : REMOVE USELESS COLUMNS
================================================================
md5    = unique file fingerprint ID  → NOT a feature, remove it
sha1   = another unique ID          → NOT a feature, remove it
Category = another version of label → keeping it = CHEATING
Family   = tells exact ransomware family → also CHEATING
We only keep columns that describe the file's behavior/structure

In [ ]:
print("\n[STEP 2]  Removing useless columns...")
print("         Dropping: md5, sha1, Category, Family")

In [ ]:
df = df.drop(columns=["md5", "sha1", "Category", "Family"], errors="ignore")

In [ ]:
print(f"         Columns remaining : {df.shape[1]}")

================================================================
STEP 3 : CREATE BINARY LABEL  ( 0 = Benign,  1 = Malicious )
================================================================
Original 'Class' column has 27 values:
  "Benign", "WannaCry", "REvil", "LockBit", "Ryuk" ... etc

We simplify to just TWO values:
  Benign      → 0   (safe file)
  Any malware → 1   (dangerous file)

Why? Our question is only "safe OR dangerous?"

In [ ]:
print("\n[STEP 3]  Creating binary label (0=Benign, 1=Malicious)...")

In [ ]:
df["label"] = (df["Class"] != "Benign").astype(int)
df = df.drop(columns=["Class"])

In [ ]:
benign_count    = (df["label"] == 0).sum()
malicious_count = (df["label"] == 1).sum()
total           = len(df)

In [ ]:
print(f"\n         Benign    (0) : {benign_count} files")
print(f"         Malicious (1) : {malicious_count} files")
print(f"         Total         : {total} files")
print(f"         Balance       : {benign_count/total*100:.1f}% / {malicious_count/total*100:.1f}%  (Benign/Malicious)")

================================================================
STEP 4 : CONVERT TEXT AND HEX VALUES TO NUMBERS
================================================================
Machine Learning models ONLY understand numbers, not text.
Your dataset has 3 types of text columns that need converting:

  TYPE A - Hex strings  like "0x00400000"
           These are numbers in base-16 (hexadecimal)
           Convert → normal integer   "0x00400000" → 4194304

  TYPE B - List strings like "['FLAG_NX', 'FLAG_DYNAMIC_BASE']"
           Convert → one integer using hash()

  TYPE C - Normal text  like "PE32", "AMD64", "Windows GUI"
           Convert using LabelEncoder:
           "PE32" → 0,  "PE32+" → 1,  "ROM" → 2

After this step: every single column is a NUMBER

In [ ]:
print("\n[STEP 4]  Converting text and hex columns to numbers...")

In [ ]:
def hex_to_int(value):
    # Convert hex string like "0x00400000" to integer 4194304
    try:
        v = re.sub(r'\s.*', '', str(value).strip())
        return int(v, 16)   # 16 = hexadecimal base
    except:
        return np.nan

In [ ]:
text_cols = [c for c in df.select_dtypes(include=["object", "string"]).columns
             if c != "label"]

In [ ]:
print(f"         Text columns found : {len(text_cols)}")

In [ ]:
HEX_PAT  = re.compile(r'^0x[0-9a-fA-F]+')
LIST_PAT = re.compile(r'^\[')

In [ ]:
for col in text_cols:
    sample = " ".join(df[col].dropna().astype(str).iloc[:5].tolist())
    if HEX_PAT.search(sample):
        df[col] = df[col].apply(lambda v: hex_to_int(v) if pd.notna(v) else np.nan)
    elif LIST_PAT.search(sample):
        df[col] = df[col].astype(str).apply(lambda x: hash(x) % (2**31))
    else:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
print(f"         All {len(text_cols)} text columns converted to numbers")

================================================================
STEP 5 : FILL MISSING VALUES WITH MEDIAN
================================================================
NaN = empty cell = Not a Number
Models cannot handle empty cells → we fill them

Why MEDIAN and not AVERAGE?
  Average is pulled by extreme values (outliers)
  Median = the actual middle value = safe and stable
  Example:  values = [10, 12, 15, 11, 50000]
  Average  = 10009  ← pulled way up by 50000
  Median   = 12     ← the real middle, much better

In [ ]:
print("\n[STEP 5]  Filling missing values with median...")

In [ ]:
missing_before = df.isnull().sum().sum()
df = df.fillna(df.median(numeric_only=True))
missing_after  = df.isnull().sum().sum()

In [ ]:
print(f"         Missing cells before : {missing_before}")
print(f"         Missing cells after  : {missing_after}")

================================================================
STEP 6 : SPLIT INTO TRAIN AND TEST SETS  ( 80% / 20% )
================================================================
We split the data into TWO completely separate parts:

  TRAINING SET (80%) → 17,401 files → model LEARNS from this
  TEST SET     (20%) → 4,351  files → model is TESTED on this

MOST IMPORTANT RULE:
  Model NEVER sees test data during training.
  Test data = simulates real world (new unknown files)

stratify=y  → ensures both sets keep the 50/50 class ratio
random_state=42 → same split every time (reproducible results)

In [ ]:
print("\n[STEP 6]  Splitting  80% Train  /  20% Test...")

In [ ]:
X = df.drop(columns=["label"])   # features = inputs to the model
y = df["label"]                   # label    = what model must predict

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,   # 20% for testing
    random_state = 42,     # fixed seed → same result every run
    stratify     = y       # keep 50/50 balance in both sets
)

In [ ]:
print(f"\n         Training samples : {len(X_train)}  (80%)")
print(f"         Test samples     : {len(X_test)}   (20%)")
print(f"         Features         : {X_train.shape[1]}")

================================================================
STEP 7 : SHOW TRAIN AND TEST DATA SAMPLES
================================================================

In [ ]:
print("\n" + "=" * 65)
print("  TRAINING DATA  —  First 5 rows")
print("=" * 65)
train_display = pd.concat([X_train.reset_index(drop=True),
                            y_train.reset_index(drop=True)], axis=1)
# Show only first 8 columns + label so it fits on screen
cols_to_show = list(X_train.columns[:8]) + ["label"]
print(train_display[cols_to_show].head(5).to_string(index=True))
print(f"\n  ... and {len(train_display)-5} more rows")
print(f"  Shape : {train_display.shape[0]} rows  x  {train_display.shape[1]} columns")

In [ ]:
print("\n" + "-" * 65)
print("  TRAINING DATA  —  Label distribution")
print("-" * 65)
tr_benign    = (y_train == 0).sum()
tr_malicious = (y_train == 1).sum()
print(f"  Benign    (0) : {tr_benign}   ({tr_benign/len(y_train)*100:.1f}%)")
print(f"  Malicious (1) : {tr_malicious}  ({tr_malicious/len(y_train)*100:.1f}%)")

In [ ]:
print("\n" + "=" * 65)
print("  TEST DATA  —  First 5 rows")
print("=" * 65)
test_display = pd.concat([X_test.reset_index(drop=True),
                           y_test.reset_index(drop=True)], axis=1)
print(test_display[cols_to_show].head(5).to_string(index=True))
print(f"\n  ... and {len(test_display)-5} more rows")
print(f"  Shape : {test_display.shape[0]} rows  x  {test_display.shape[1]} columns")

In [ ]:
print("\n" + "-" * 65)
print("  TEST DATA  —  Label distribution")
print("-" * 65)
te_benign    = (y_test == 0).sum()
te_malicious = (y_test == 1).sum()
print(f"  Benign    (0) : {te_benign}   ({te_benign/len(y_test)*100:.1f}%)")
print(f"  Malicious (1) : {te_malicious}  ({te_malicious/len(y_test)*100:.1f}%)")

================================================================
STEP 8 : SAVE train.csv AND test.csv
================================================================

In [ ]:
print("\n[STEP 8]  Saving CSV files...")

In [ ]:
train_display.to_csv("data/train.csv", index=False)
test_display.to_csv("data/test.csv",   index=False)

In [ ]:
print(f"         train.csv saved  →  data/train.csv")
print(f"         test.csv  saved  →  data/test.csv")

================================================================
STEP 9 : SAVE CLASS DISTRIBUTION PLOT
================================================================

In [ ]:
print("\n[STEP 9]  Saving class distribution plot...")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

In [ ]:
# Plot 1 — Full dataset
axes[0].bar(["Benign", "Malicious"], [benign_count, malicious_count],
            color=["steelblue", "tomato"], edgecolor="black", width=0.5)
axes[0].set_title("Full Dataset", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Files")
for i, v in enumerate([benign_count, malicious_count]):
    axes[0].text(i, v + 80, str(v), ha="center", fontweight="bold")

In [ ]:
# Plot 2 — Training set
axes[1].bar(["Benign", "Malicious"], [tr_benign, tr_malicious],
            color=["steelblue", "tomato"], edgecolor="black", width=0.5)
axes[1].set_title("Training Set  (80%)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Number of Files")
for i, v in enumerate([tr_benign, tr_malicious]):
    axes[1].text(i, v + 80, str(v), ha="center", fontweight="bold")

In [ ]:
# Plot 3 — Test set
axes[2].bar(["Benign", "Malicious"], [te_benign, te_malicious],
            color=["steelblue", "tomato"], edgecolor="black", width=0.5)
axes[2].set_title("Test Set  (20%)", fontsize=13, fontweight="bold")
axes[2].set_ylabel("Number of Files")
for i, v in enumerate([te_benign, te_malicious]):
    axes[2].text(i, v + 80, str(v), ha="center", fontweight="bold")

In [ ]:
plt.suptitle("Class Distribution — Full / Train / Test", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/plots/class_distribution.png", dpi=150)
plt.close()
print(f"         Plot saved  →  results/plots/class_distribution.png")

In [ ]:
# ================================================================
# FINAL SUMMARY — printed below the code
# ================================================================
print("\n" + "=" * 65)
print("  PREPROCESSING COMPLETE — FULL SUMMARY")
print("=" * 65)
print(f"  Raw file loaded    :  data/Final_Dataset_without_duplicate.csv")
print(f"  Total .exe files   :  {total}")
print(f"  Features kept      :  {X.shape[1]}")
print(f"  Missing values     :  {missing_before} filled with median")
print()
print(f"  FULL DATASET")
print(f"    Benign    (0)    :  {benign_count}  ({benign_count/total*100:.1f}%)")
print(f"    Malicious (1)    :  {malicious_count}  ({malicious_count/total*100:.1f}%)")
print(f"    Balance          :  Perfect 50/50 — SMOTE not needed")
print()
print(f"  TRAINING SET  (80%)")
print(f"    Total rows       :  {len(X_train)}")
print(f"    Benign    (0)    :  {tr_benign}")
print(f"    Malicious (1)    :  {tr_malicious}")
print(f"    Saved to         :  data/train.csv")
print()
print(f"  TEST SET  (20%)")
print(f"    Total rows       :  {len(X_test)}")
print(f"    Benign    (0)    :  {te_benign}")
print(f"    Malicious (1)    :  {te_malicious}")
print(f"    Saved to         :  data/test.csv")
print()
print(f"  Plot saved         :  results/plots/class_distribution.png")
print("=" * 65)
print("  NEXT STEP  →  Run  train.py")
print("=" * 65)